In [9]:
# conda activate genomic_tools

import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## Append InterProScan data to exon results

In [10]:
cols = [
    'protein_accession',
    'sequence_md5',
    'sequence_length',
    'analysis',
    'signature_accession',
    'signature_description',
    'start',
    'stop',
    'score',
    'status',
    'date',
    'interpro_accession',
    'interpro_description',
    'go_annotations',
    'pathways'
]

interpro_df = pd.read_csv(
    "data/interproscan/proteins_filtered.fa.tsv",
    sep='\t',
    header=None,
    names=cols,
    index_col=False
)

In [11]:
# Parse the PIRSR data

import json

with open("data/interproscan/sr_uru.json") as f:
    pirsr_data = json.load(f)

# Subset to proteins patterns applicable to humans
human_relevant = ['Eukaryota', 'Eukaryota; Metazoa', 'Eukaryota; Vertebrata', 'Eukaryota; Chordata', 'Eukaryota; Mammalia', 'Eukaryota; Eutheria']

records = []
for ac, entry in pirsr_data.items():
    for group_id, sites in entry['Groups'].items():
        for site in sites:
            scope = entry.get('Scope', [])
            tr = entry.get('TR', '')
            if any(s in human_relevant for s in scope):
                records.append({
                    'accession': ac,
                    'scope':  ', '.join(scope),
                    'TR': tr.split("; ")[1],
                    'label': site['label'],
                    'condition': site['condition'],
                    'desc': site['desc'],
                    'group': group_id
                })

pirsr_df = pd.DataFrame(records)

In [12]:
# note: within a single group, all sites must be present in the same protein for the rule to fire
pirsr_df = pirsr_df.groupby("accession").agg(
    scope=('scope', lambda x: ' | '.join(x.unique())),
    TR=('TR', lambda x: ' | '.join(x.unique())),
    label=('label', lambda x: ' | '.join(x.unique())),
    condition=('condition', lambda x: ' | '.join(x.unique())),
    desc=('desc', lambda x: ' | '.join(x.unique())),
    group=('group', lambda x: ' | '.join(x.unique())),
).reset_index()

In [13]:
interpro_df = interpro_df.merge(pirsr_df, left_on="signature_accession", right_on="accession", how="left")

In [18]:
interpro_df.head()

,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,stop,score,status,date,interpro_accession,interpro_description,go_annotations,pathways,accession,scope,TR,label,condition,desc,group
0,ENST00000591522,C49E70B1DDFB8726EF72822A2C1687CA,341,PIRSR,PIRSR001174-2,-,146,176,8.9E-5,PIRSR,05-06-2026,-,-,-,-,PIRSR001174-2,"Eukaryota, Bacteria, Archaea",PIRSF001174,NP_BIND,G-x(6)-[TS],ATP.,1
1,ENST00000591522,C49E70B1DDFB8726EF72822A2C1687CA,341,Pfam,PF00004,ATPase family associated with various cellular...,146,260,4.2E-28,Pfam,05-06-2026,IPR003959,"ATPase, AAA-type, core",GO:0005524(InterPro)|GO:0016887(InterPro),-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ENST00000591522,C49E70B1DDFB8726EF72822A2C1687CA,341,SMART,SM00382,ATPases associated with a variety of cellular ...,142,282,2.0E-13,SMART,05-06-2026,IPR003593,AAA+ ATPase domain,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ENST00000591522,C49E70B1DDFB8726EF72822A2C1687CA,341,SUPERFAMILY,SSF52540,P-loop containing nucleoside triphosphate hydr...,112,260,1.5E-32,SUPERFAMILY,05-06-2026,IPR027417,P-loop containing nucleoside triphosphate hydr...,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ENST00000695547,C4A5672158C594A15D70CFF6500A7D33,561,COILS,Coil,Coil,398,418,-,COILS,05-06-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
signif_exons.head()

,event,event,Gene,is_specific,specific_direction,chr,strand,exon_start,exon_end,exon_len,transcript,transcript_type,exon_number,tag,coding_nt_length,full_exon_nt_length,overlap_type,aa_start,aa_end,RSEM_detected,RSEM_mean_expr,RSEM_expr_corr,r,fdr,CGE Class,All GABAergic,All Neuronal,Upper layer glutamatergic,Deep layer glutamatergic,Oligo,OPC,Astro,Micro/PVM,VLMC,Endo,Peri,r_diff_CGE Class,fdr_diff_CGE Class,r_diff_All GABAergic,fdr_diff_All GABAergic,r_diff_All Neuronal,fdr_diff_All Neuronal,r_diff_Upper layer glutamatergic,fdr_diff_Upper layer glutamatergic,r_diff_Deep layer glutamatergic,fdr_diff_Deep layer glutamatergic,r_diff_OPC,fdr_diff_OPC,r_diff_Astro,fdr_diff_Astro,r_diff_Micro/PVM,fdr_diff_Micro/PVM,r_diff_VLMC,fdr_diff_VLMC,r_diff_Endo,fdr_diff_Endo,r_diff_Peri,fdr_diff_Peri,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,stop,score,status,date,interpro_accession,interpro_description,go_annotations,pathways,accession,scope,TR,label,condition,desc,group,overlapping
0,0,ENSG00000107331_ProteinCoding_2,ABCA2,True,highest,chr9,-,137023838,137023840,3,ENST00000341511,protein_coding,3,"basic,Ensembl_canonical,GENCODE_Primary,MANE_S...",3.0,3,fully_coding,53.0,54.0,True,15.808164,0.621401,0.676605,0.0,-0.634909,-0.610043,-0.435386,-0.499214,0.093277,0.676605,0.125003,0.204347,-0.040752,0.254265,0.404077,0.277828,1.311513,0.0,1.286648,0.0,1.11199,0.0,1.175818,0.0,0.583328,0.0,0.551601,0.0,0.472258,0.0,0.717356,0.0,0.422339,0.0,0.272528,0.0,0.398776,0.0,ENST00000341511,D0B988647F93F4423C42C776F903FB63,2436.0,COILS,Coil,Coil,1097.0,1117.0,-,COILS,05-06-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1,ENSG00000107331_ProteinCoding_2,ABCA2,True,highest,chr9,-,137023838,137023840,3,ENST00000341511,protein_coding,3,"basic,Ensembl_canonical,GENCODE_Primary,MANE_S...",3.0,3,fully_coding,53.0,54.0,True,15.808164,0.621401,0.676605,0.0,-0.634909,-0.610043,-0.435386,-0.499214,0.093277,0.676605,0.125003,0.204347,-0.040752,0.254265,0.404077,0.277828,1.311513,0.0,1.286648,0.0,1.11199,0.0,1.175818,0.0,0.583328,0.0,0.551601,0.0,0.472258,0.0,0.717356,0.0,0.422339,0.0,0.272528,0.0,0.398776,0.0,ENST00000341511,D0B988647F93F4423C42C776F903FB63,2436.0,PROSITE patterns,PS00211,ABC transporters family signature,1124.0,1138.0,-,PROSITE patterns,05-06-2026,IPR017871,"ABC transporter-like, conserved site",GO:0005524(InterPro)|GO:0016887(InterPro),-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,2,ENSG00000107331_ProteinCoding_2,ABCA2,True,highest,chr9,-,137023838,137023840,3,ENST00000341511,protein_coding,3,"basic,Ensembl_canonical,GENCODE_Primary,MANE_S...",3.0,3,fully_coding,53.0,54.0,True,15.808164,0.621401,0.676605,0.0,-0.634909,-0.610043,-0.435386,-0.499214,0.093277,0.676605,0.125003,0.204347,-0.040752,0.254265,0.404077,0.277828,1.311513,0.0,1.286648,0.0,1.11199,0.0,1.175818,0.0,0.583328,0.0,0.551601,0.0,0.472258,0.0,0.717356,0.0,0.422339,0.0,0.272528,0.0,0.398776,0.0,ENST00000341511,D0B988647F93F4423C42C776F903FB63,2436.0,PROSITE profiles,PS50893,"ATP-binding cassette, ABC transporter-type dom...",991.0,1222.0,18.740957,PROSITE profiles,05-06-2026,IPR003439,"ABC transporter-like, ATP-binding domain",GO:0005524(InterPro)|GO:0016887(InterPro),-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
3,3,ENSG00000107331_ProteinCoding_2,ABCA2,True,highest,chr9,-,137023838,137023840,3,ENST00000341511,protein_coding,3,"basic,Ensembl_canonical,GENCODE_Primary,MANE_S...",3.0,3,fully_coding,53.0,54.0,True,15.808164,0.621401,0.676605,0.0,-0.634909,-0.610043,-0.435386,-0.499214,0.093277,0.676605,0.125003,0.204347,-0.040752,0.254265,0.404077,0.277828,1.311513,0.0,1.286648,0.0,1.11199,0.0,1.175818,0.0,0.583328,0.0,0.551601,0.0,0.472258,0.0,0.717356,0.0,0.422339,0.0,0.272528,0.0,0.398776,0.0,ENST00000341511,D0B988647F93F4423C42C776F903FB63,2436.0,PROSITE profiles,PS50893,"ATP-binding cassette, ABC transporter-type dom...",2051.0,2286.0,18.747625,PROSITE profiles,05-06-2026,IPR003439,"ABC transporter-like, ATP-binding domain",GO:000

In [ ]:
def residue_ownership(p, aa_start, aa_end, clean_start, clean_end):
    if aa_start < p < aa_end:
        return "interior"          # codon fully inside the exon — clean toggle
    if p == aa_start:
        return "clean" if clean_start else "boundary_shared"
    if p == aa_end:
        return "clean" if clean_end else "boundary_shared"
    return "outside"               # feature didn't actually fall in the exon

In [20]:
for file in os.listdir("data/ctype_exons/annotated"):
    if file.endswith("annotated.csv"):
        signif_exons = pd.read_csv(f"data/ctype_exons/annotated/{file}")
        signif_exons = signif_exons.rename(columns={signif_exons.columns[0]: "event"})
        
        exons_interpro_df = signif_exons.merge(interpro_df, left_on="transcript", right_on="protein_accession", how="left")
        
        # indicate domain hits that overlap exon
        overlap = (exons_interpro_df['start'] <= exons_interpro_df['aa_end']) & (exons_interpro_df['stop'] >= exons_interpro_df['aa_start'])
        analyses = ['NCBIFAM', 'SFLD'] # these tools are for full-length protein classification
        mask = overlap & ~(exons_interpro_df['analysis'].isin(analyses))
        exons_interpro_df['overlapping'] = False
        exons_interpro_df.loc[mask, 'overlapping'] = True

        exons_interpro_df.to_csv(f"data/ctype_exons/annotated/{file.replace('.csv', '')}_interproscan.csv")

In [27]:
signif_exons.head()

,event,Gene,is_specific,specific_direction,chr,strand,exon_start,exon_end,exon_len,transcript,transcript_type,exon_number,tag,coding_nt_length,full_exon_nt_length,overlap_type,aa_start,aa_end,RSEM_detected,RSEM_mean_expr,RSEM_expr_corr,r,fdr,CGE Class,All GABAergic,All Neuronal,Upper layer glutamatergic,Deep layer glutamatergic,Oligo,OPC,Astro,Micro/PVM,VLMC,Endo,Peri,r_diff_CGE Class,fdr_diff_CGE Class,r_diff_All GABAergic,fdr_diff_All GABAergic,r_diff_All Neuronal,fdr_diff_All Neuronal,r_diff_Deep layer glutamatergic,fdr_diff_Deep layer glutamatergic,r_diff_Oligo,fdr_diff_Oligo,r_diff_OPC,fdr_diff_OPC,r_diff_Astro,fdr_diff_Astro,r_diff_Micro/PVM,fdr_diff_Micro/PVM,r_diff_VLMC,fdr_diff_VLMC,r_diff_Endo,fdr_diff_Endo,r_diff_Peri,fdr_diff_Peri
0,ENSG00000107863_ProteinCoding_1,ARHGAP21,False,NaN,chr10,-,24590206,24590479,274,ENST00000636789,protein_coding,18,"CAGE_supported_TSS,NMD_exception,basic,GENCODE...",140.0,274,partially_coding,1170.0,1216.0,True,6.265549,0.571558,0.703054,0.0,0.476509,0.490460,0.723407,0.703054,0.242230,-0.326878,-0.554642,-0.625831,-0.064554,-0.415744,-0.391791,-0.451343,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ENSG00000111907_ProteinCoding_1,TPD52L1,False,NaN,chr6,+,125252022,125252036,15,ENST00000304877,protein_coding,5,"basic,GENCODE_Primary,appris_alternative_1,CCDS",15.0,15,fully_coding,128.0,133.0,True,8.121118,0.650607,0.694785,0.0,0.579044,0.621569,0.665763,0.694785,0.271378,-0.261962,-0.405426,-0.621354,-0.051557,-0.381681,-0.508641,-0.407914,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ENSG00000197555_ProteinCoding_3,SIPA1L1,False,NaN,chr14,+,71704721,71704783,63,ENST00000555818,protein_coding,13,"basic,CCDS",63.0,63,fully_coding,1215.0,1236.0,True,13.983134,0.637105,0.683839,0.0,0.368978,0.418333,0.705225,0.683839,0.335319,-0.177228,-0.539327,-0.618277,-0.075260,-0.412460,-0.406790,-0.431271,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ENSG00000148341_ProteinCoding_1,SH3GLB2,False,NaN,chr9,-,129009453,129009467,15,ENST00000479237,retained_intron,4,NaN,NaN,15,NaN,NaN,NaN,True,15.087685,0.054001,0.676214,0.0,0.608593,0.622275,0.651904,0.676214,0.095309,-0.552310,-0.333363,-0.402066,-0.060191,-0.449591,-0.504741,-0.462548,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ENSG00000148341_ProteinCoding_1,SH3GLB2,False,NaN,chr9,-,129009453,129009467,15,ENST00000372554,protein_coding,11,"basic,GENCODE_Primary,appris_alternative_1,CCDS",15.0,15,fully_coding,283.0,288.0,True,79.055649,0.504872,0.676214,0.0,0.608593,0.622275,0.651904,0.676214,0.095309,-0.552310,-0.333363,-0.402066,-0.060191,-0.449591,-0.504741,-0.462548,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Manually inspect results

In [22]:
exon_df = pd.read_csv("data/ctype_exons/annotated/Upper_layer_glutamatergic_exons_annotated_interproscan.csv", index_col=0, low_memory=False)

In [24]:
exon_df.head()

,event,Gene,is_specific,specific_direction,chr,strand,exon_start,exon_end,exon_len,transcript,transcript_type,exon_number,tag,coding_nt_length,full_exon_nt_length,overlap_type,aa_start,aa_end,RSEM_detected,RSEM_mean_expr,RSEM_expr_corr,r,fdr,CGE Class,All GABAergic,All Neuronal,Upper layer glutamatergic,Deep layer glutamatergic,Oligo,OPC,Astro,Micro/PVM,VLMC,Endo,Peri,r_diff_CGE Class,fdr_diff_CGE Class,r_diff_All GABAergic,fdr_diff_All GABAergic,r_diff_All Neuronal,fdr_diff_All Neuronal,r_diff_Deep layer glutamatergic,fdr_diff_Deep layer glutamatergic,r_diff_Oligo,fdr_diff_Oligo,r_diff_OPC,fdr_diff_OPC,r_diff_Astro,fdr_diff_Astro,r_diff_Micro/PVM,fdr_diff_Micro/PVM,r_diff_VLMC,fdr_diff_VLMC,r_diff_Endo,fdr_diff_Endo,r_diff_Peri,fdr_diff_Peri,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,stop,score,status,date,interpro_accession,interpro_description,go_annotations,pathways,accession,scope,TR,label,condition,desc,group,overlapping
0,ENSG00000107863_ProteinCoding_1,ARHGAP21,False,NaN,chr10,-,24590206,24590479,274,ENST00000636789,protein_coding,18,"CAGE_supported_TSS,NMD_exception,basic,GENCODE...",140.0,274,partially_coding,1170.0,1216.0,True,6.265549,0.571558,0.703054,0.0,0.476509,0.49046,0.723407,0.703054,0.24223,-0.326878,-0.554642,-0.625831,-0.064554,-0.415744,-0.391791,-0.451343,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENST00000636789,4BBEFBE171CC8D07642FDC4DFD8C1CC7,1217.0,COILS,Coil,Coil,838.0,858.0,-,COILS,05-06-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,ENSG00000107863_ProteinCoding_1,ARHGAP21,False,NaN,chr10,-,24590206,24590479,274,ENST00000636789,protein_coding,18,"CAGE_supported_TSS,NMD_exception,basic,GENCODE...",140.0,274,partially_coding,1170.0,1216.0,True,6.265549,0.571558,0.703054,0.0,0.476509,0.49046,0.723407,0.703054,0.24223,-0.326878,-0.554642,-0.625831,-0.064554,-0.415744,-0.391791,-0.451343,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENST00000636789,4BBEFBE171CC8D07642FDC4DFD8C1CC7,1217.0,PROSITE profiles,PS50238,Rho GTPase-activating proteins domain profile,934.0,1126.0,49.701973,PROSITE profiles,05-06-2026,IPR000198,Rho GTPase-activating protein domain,GO:0007165(InterPro),-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,ENSG00000107863_ProteinCoding_1,ARHGAP21,False,NaN,chr10,-,24590206,24590479,274,ENST00000636789,protein_coding,18,"CAGE_supported_TSS,NMD_exception,basic,GENCODE...",140.0,274,partially_coding,1170.0,1216.0,True,6.265549,0.571558,0.703054,0.0,0.476509,0.49046,0.723407,0.703054,0.24223,-0.326878,-0.554642,-0.625831,-0.064554,-0.415744,-0.391791,-0.451343,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENST00000636789,4BBEFBE171CC8D07642FDC4DFD8C1CC7,1217.0,PROSITE profiles,PS50003,PH domain profile,718.0,827.0,14.6862,PROSITE profiles,05-06-2026,IPR001849,Pleckstrin homology domain,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
3,ENSG00000107863_ProteinCoding_1,ARHGAP21,False,NaN,chr10,-,24590206,24590479,274,ENST00000636789,protein_coding,18,"CAGE_supported_TSS,NMD_exception,basic,GENCODE...",140.0,274,partially_coding,1170.0,1216.0,True,6.265549,0.571558,0.703054,0.0,0.476509,0.49046,0.723407,0.703054,0.24223,-0.326878,-0.554642,-0.625831,-0.064554,-0.415744,-0.391791,-0.451343,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENST00000636789,4BBEFBE171CC8D07642FDC4DFD8C1CC7,1217.0,Pfam,PF00620,RhoGAP domain,949.0,1097.0,2.2E-45,Pfam,05-06-2026,IPR000198,Rho GTPase-activating protein domain,GO:0007165(InterPro),-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4,ENSG00000107863_ProteinCoding_1,ARHGAP21,False,NaN,chr10,-,24590206,24590479,274,ENST00000636789,protein_coding,18,"CAGE_supported_TSS,NMD_exception,basic,GENCODE...",140.0,274,partially_coding,1170.0,1216.0,True,6.265549,0.571558,0.703054,0.0,0.476509,0.49046,0.723407,0.703054,0.24223,-0.326878,-0.554642,-0.625831,-0.064554,-0.41574

In [26]:
mask = exon_df['overlapping'] == True

cols = ['event', 'Gene', 'chr', 'exon_start', 'exon_end', 'transcript', 'specific_direction', 'coding_nt_length', 'aa_start', 'aa_end', 'start', 'stop', 'analysis', 'signature_accession', 'interpro_accession', 'signature_description', 'interpro_description', 'label', 'desc']

exon_df.loc[mask, cols].sort_values(["specific_direction", "event", "transcript"]).head(10)

,event,Gene,chr,exon_start,exon_end,transcript,specific_direction,coding_nt_length,aa_start,aa_end,start,stop,analysis,signature_accession,interpro_accession,signature_description,interpro_description,label,desc
57288,ENSG00000068120_ProteinCoding_1,COASY,chr17,42562296,42562487,ENST00000587858,highest,76.0,0.0,25.0,1.0,52.0,DeepTMHMM,Signal Peptide,-,-,-,NaN,NaN
66100,ENSG00000075043_ProteinCoding_2,KCNQ2,chr20,63411775,63411882,ENST00000706989,highest,108.0,569.0,605.0,448.0,595.0,Pfam,PF03520,IPR013821,KCNQ voltage-gated potassium channel,"Potassium channel, voltage dependent, KCNQ, C-...",NaN,NaN
66101,ENSG00000075043_ProteinCoding_2,KCNQ2,chr20,63411775,63411882,ENST00000706989,highest,108.0,569.0,605.0,605.0,667.0,Pfam,PF03520,IPR013821,KCNQ voltage-gated potassium channel,"Potassium channel, voltage dependent, KCNQ, C-...",NaN,NaN
66111,ENSG00000075043_ProteinCoding_2,KCNQ2,chr20,63411775,63411882,ENST00000706989,highest,108.0,569.0,605.0,319.0,890.0,Phobius,CYTOPLASMIC_DOMAIN,-,Cytoplasmic domain,-,NaN,NaN
49549,ENSG00000103197_ProteinCoding_5,TSC2,chr16,2077598,2077726,ENST00000219476,highest,129.0,945.0,988.0,899.0,1807.0,Phobius,CYTOPLASMIC_DOMAIN,-,Cytoplasmic domain,-,NaN,NaN
69750,ENSG00000114841_ProteinCoding_3,DNAH1,chr3,52398032,52398162,ENST00000420323,highest,131.0,3986.0,4029.0,3959.0,4261.0,Pfam,PF18199,IPR041228,Dynein heavy chain C-terminal domain,"Dynein heavy chain, C-terminal domain",NaN,NaN
33699,ENSG00000115756_ProteinCoding_3,HPCAL1,chr2,10424525,10424613,ENST00000422133,highest,86.0,73.0,102.0,10.0,73.0,Pfam,PF13499,IPR002048,EF-hand domain pair,EF-hand domain,NaN,NaN
33701,ENSG00000115756_ProteinCoding_3,HPCAL1,chr2,10424525,10424613,ENST00000422133,highest,86.0,73.0,102.0,2.0,73.0,SUPERFAMILY,SSF47473,IPR011992,EF-hand,EF-hand domain pair,NaN,NaN
41684,ENSG00000121905_ProteinCoding_1,HPCA,chr1,32888878,32889276,ENST00000373467,highest,378.0,0.0,125.0,76.0,170.0,PIRSR,PIRSR608080-1,-,-,-,BINDING,Ca(2+)
41685,ENSG00000121905_ProteinCoding_1,HPCA,chr1,32888878,32889276,ENST00000373467,highest,378.0,0.0,125.0,73.0,85.0,PROSITE patterns,PS00018,IPR018247,EF-hand calcium-binding domain,"EF-Hand 1, calcium-binding site",NaN,NaN


In [ ]:
temp = exon_df.loc[mask, cols].sort_values(["specific_direction", "event", "transcript"])

In [ ]:
temp[temp['event'] == "ENSG00000157388_ProteinCoding_6"]

,event,Gene,chr,exon_start,exon_end,transcript,specific_direction,coding_nt_length,aa_start,aa_end,start,stop,analysis,signature_accession,interpro_accession,signature_description,interpro_description,label,desc
28988,ENSG00000157388_ProteinCoding_6,CACNA1D,chr3,53762551,53762634,ENST00000640483,NaN,84.0,1273.0,1300.0,838.0,1613.0,PIRSR,PIRSR602077-1,-,-,-,BINDING,Ca(2+)
28994,ENSG00000157388_ProteinCoding_6,CACNA1D,chr3,53762551,53762634,ENST00000640483,NaN,84.0,1273.0,1300.0,1216.0,1470.0,Pfam,PF00520,IPR005821,Ion transport protein,Ion transport domain,NaN,NaN
29004,ENSG00000157388_ProteinCoding_6,CACNA1D,chr3,53762551,53762634,ENST00000640483,NaN,84.0,1273.0,1300.0,1269.0,1279.0,Phobius,CYTOPLASMIC_DOMAIN,-,Cytoplasmic domain,-,NaN,NaN
29016,ENSG00000157388_ProteinCoding_6,CACNA1D,chr3,53762551,53762634,ENST00000640483,NaN,84.0,1273.0,1300.0,1298.0,1308.0,Phobius,NON_CYTOPLASMIC_DOMAIN,-,Non cytoplasmic domain,-,NaN,NaN
29036,ENSG00000157388_ProteinCoding_6,CACNA1D,chr3,53762551,53762634,ENST00000640483,NaN,84.0,1273.0,1300.0,1280.0,1297.0,Phobius,TRANSMEMBRANE,-,Transmembrane region,-,NaN,NaN
29044,ENSG00000157388_ProteinCoding_6,CACNA1D,chr3,53762551,53762634,ENST00000640483,NaN,84.0,1273.0,1300.0,1217.0,1463.0,SUPERFAMILY,SSF81324,-,Voltage-gated potassium channels,-,NaN,NaN
29065,ENSG00000157388_ProteinCoding_6,CACNA1D,chr3,53762551,53762634,ENST00000640483,NaN,84.0,1273.0,1300.0,1285.0,1297.0,DeepTMHMM,Transmembrane alpha helix,-,-,-,NaN,NaN


In [8]:
temp_group['exon_id'] = temp["chr"].str + temp["exon_start"].str + temp['exon_id'].str

NameError: name 'temp' is not defined

In [ ]:
temp_group = temp.groupby(["chr", "exon_start", "exon_end"])

In [ ]:
temp_group

In [ ]:

# Re-aggregate to the exon, then check concordance. 
# You already have the genomic coordinates (chr, exon_start, exon_end), so collapse all events onto the exon (that's the exon_id I built above) 
# and look at the direction of every event touching it. If all events on an exon move the same way, you have a clean "this exon is regulated" story regardless of how many junctions express it. If two events on the same exon point opposite ways, that's the signature of a junction-context effect rather than simple inclusion/skipping — the exon isn't going up or down, the partner is changing. This is the one step you can do in your file today; the blocker is that direction is populated on almost none of your rows, so you'll need to merge the upstream quantification table back on event first.